# Proyecto Final - Análisis de Datos 
# Formula 1
### 04 - Análisis SQL
**Integrantes:** 
- Esquivel, Kevin
- Simmons, Abigail
- Solis, Luis
- Villarreal, Sergio

**Fecha:** Julio 2026  

En este notebook complementamos la estadística descriptiva del notebook `03_analisis_exploratorio`
usando **consultas SQL** directamente sobre las tablas registradas en `01_ingesta_datos` y
`02_limpieza_datos`, y sobre las tablas resumen que ya dejamos guardadas en el notebook 03
(`puntos_por_escuderia`, `resumen_escuderia`, etc.).

## 1. Top 10 pilotos por puntos totales acumulados

In [0]:
-- Top 10 pilotos por puntos totales acumulados en toda su carrera
SELECT
  p.nombre_piloto,
  p.apellido_piloto,
  p.nacionalidad,
  COUNT(r.id_carrera)      AS carreras_disputadas,
  SUM(r.puntos_obtenidos)  AS puntos_totales,
  ROUND(AVG(r.posicion_final), 2) AS posicion_promedio
FROM resultado r
JOIN piloto p ON p.id_piloto = r.id_piloto
GROUP BY p.nombre_piloto, p.apellido_piloto, p.nacionalidad
ORDER BY puntos_totales DESC
LIMIT 10;

nombre_piloto,apellido_piloto,nacionalidad,carreras_disputadas,puntos_totales,posicion_promedio
Daniel,Ricciardo,Australia,59,412,9.08
Juan,Rodriguez,Mexico,50,383,9.0
Franco,Colapinto,Argentina,56,319,10.09
Charles,Leclerc,Monaco,45,314,8.78
Max,Verstappen,Netherlands,48,304,10.35
Andrea,Kim,Canada,57,304,10.72
David,Coulthard,United Kingdom,44,292,9.11
Valtteri,Bottas,Finland,55,292,10.82
Alain,Prost,France,45,290,8.73
Lucas,Fontaine,France,49,289,10.1


## 2. Puntos promedio por nacionalidad de piloto

In [0]:
-- Que tan bien puntuan, en promedio, los pilotos segun su nacionalidad
SELECT
  p.nacionalidad,
  COUNT(DISTINCT p.id_piloto) AS cantidad_pilotos,
  ROUND(AVG(r.puntos_obtenidos), 3) AS puntos_promedio_por_resultado,
  SUM(r.puntos_obtenidos) AS puntos_totales
FROM resultado r
JOIN piloto p ON p.id_piloto = r.id_piloto
GROUP BY p.nacionalidad
HAVING COUNT(DISTINCT p.id_piloto) >= 3
ORDER BY puntos_promedio_por_resultado DESC;

nacionalidad,cantidad_pilotos,puntos_promedio_por_resultado,puntos_totales
France,4,5.778,1017
Brazil,5,5.311,1179
Finland,3,5.063,719
Italy,3,4.892,636
United Kingdom,13,4.623,2903
Spain,4,4.508,861
Germany,4,4.141,820


## 3. Patrocinio total recibido por sector de industria

In [0]:
-- Que sectores de industria invierten mas dinero en patrocinar escuderias de F1
SELECT
  pa.sector_industria,
  COUNT(DISTINCT pa.id_patrocinador) AS cantidad_patrocinadores,
  ROUND(SUM(f.monto_aporte), 2)      AS monto_total_usd,
  ROUND(AVG(f.monto_aporte), 2)      AS monto_promedio_usd
FROM financiamiento f
JOIN patrocinador pa ON pa.id_patrocinador = f.id_patrocinador
GROUP BY pa.sector_industria
ORDER BY monto_total_usd DESC;

sector_industria,cantidad_patrocinadores,monto_total_usd,monto_promedio_usd
Seguros,9,9.0971284285E8,3.638851371E7
Banca,8,8.3938777405E8,3.815398973E7
Bebidas,8,7.883469087E8,3.427595255E7
Telecomunicaciones,8,7.446232269E8,3.54582489E7
Automotriz,6,5.914814644E8,3.696759153E7
Tecnologia,6,5.84693596E8,4.176382829E7
Petroleo,5,4.454026956E8,3.181447826E7


## 4. Condiciones climáticas promedio por circuito

In [0]:
-- Los 10 circuitos con mayor probabilidad de lluvia promedio en sus carreras
SELECT
  c.nombre       AS circuito,
  c.ubicacion,
  COUNT(ca.id_carrera)                    AS carreras_realizadas,
  ROUND(AVG(ca.temperatura), 3)           AS temperatura_promedio_c,
  ROUND(AVG(ca.probabilidad_lluvia), 3)   AS lluvia_promedio_pct,
  ROUND(AVG(ca.cantidad_vueltas), 1)      AS vueltas_promedio
FROM carrera ca
JOIN circuito c ON c.id_circuito = ca.id_circuito
GROUP BY c.nombre, c.ubicacion
ORDER BY lluvia_promedio_pct DESC
LIMIT 10;

circuito,ubicacion,carreras_realizadas,temperatura_promedio_c,lluvia_promedio_pct,vueltas_promedio
Marina Bay,Singapur,10,31.99,65.604,56.4
Spa-Francorchamps,Belgica,10,22.75,54.003,63.6
Suzuka,Japon,20,22.37,44.018,53.7
Interlagos,Brasil,10,30.66,39.628,61.2
Hermanos Rodriguez,Mexico,10,27.52,35.798,68.8
Jerez,Espana,10,27.65,35.05,56.7
Silverstone,Reino Unido,10,22.17,25.381,64.0
Barcelona-Cataluna,Espana,10,28.43,24.038,68.1
Watkins Glen,Estados Unidos,10,24.66,19.781,61.3
Spielberg,Austria,10,17.96,12.267,72.2


## 5. Escuderías con pilotos multi-campeones

In [0]:
-- Escuderias cuyos pilotos, en conjunto, acumulan mas campeonatos
SELECT
  e.nombre AS escuderia,
  e.pais_origen,
  e.director,
  COUNT(p.id_piloto)                 AS cantidad_pilotos,
  SUM(p.cantidad_campeonatos)        AS campeonatos_acumulados,
  ROUND(AVG(p.cantidad_campeonatos), 3) AS campeonatos_promedio
FROM piloto p
JOIN escuderia e ON e.id_escuderia = p.id_escuderia
GROUP BY e.nombre, e.pais_origen, e.director
HAVING SUM(p.cantidad_campeonatos) > 0
ORDER BY campeonatos_acumulados DESC;

escuderia,pais_origen,director,cantidad_pilotos,campeonatos_acumulados,campeonatos_promedio
Ferrari,Italia,Frederic Vasseur,9,11,1.222
McLaren,Reino Unido,Andrea Stella,7,9,1.286
Red Bull,Austria,Christian Horner,5,8,1.6
Mercedes,Alemania,Toto Wolff,3,7,2.333
Lotus,Reino Unido,Colin Chapman,2,3,1.5
Brabham,Reino Unido,Jack Brabham,2,3,1.5
Cooper,Reino Unido,John Cooper,2,3,1.5
BRM,Reino Unido,Tony Rudd,2,2,1.0
Renault,Francia,Flavio Briatore,2,2,1.0
Benetton,Italia,Flavio Briatore,2,1,0.5


## 6. Cruce con el resumen del notebook 03: eficiencia por dólar invertido

Usamos la tabla `resumen_escuderia` guardada en el notebook 03 (presupuesto, puntos
totales y patrocinio ya calculados) para obtener un indicador nuevo: cuántos puntos consigue cada
escudería por cada millón de dólares de presupuesto.

In [0]:
-- Que tan eficiente es cada escuderia generando puntos por cada millon de dolares de presupuesto
SELECT
  nombre AS escuderia,
  ROUND(presupuesto / 1000000, 2)                         AS presupuesto_millones_usd,
  puntos_obtenidos                                         AS puntos_totales,
  ROUND(puntos_obtenidos / (presupuesto / 1000000), 3)     AS puntos_por_millon_usd
FROM resumen_escuderia
WHERE presupuesto > 0
ORDER BY puntos_por_millon_usd DESC;

escuderia,presupuesto_millones_usd,puntos_totales,puntos_por_millon_usd
Ferrari,150.98,1884.0,12.479
Williams,334.79,1124.0,3.357
McLaren,558.25,1749.0,3.133
Renault,311.9,838.0,2.687
Alfa Romeo,367.86,826.0,2.245
Brabham,206.06,408.0,1.98
Lotus,429.27,748.0,1.742
Tyrrell,302.05,522.0,1.728
Red Bull,354.79,568.0,1.601
Toro Rosso,311.32,486.0,1.561


## 7. Resumen de hallazgos

- Se identificó el **top de pilotos** por puntos totales y su posición promedio de llegada.
- Se comparó el **rendimiento promedio por nacionalidad** de piloto (con un mínimo de 3 pilotos
  para evitar sesgos).
- Se cuantificó el **patrocinio recibido por sector de industria**, mostrando qué sectores
  invierten más en F1.
- Se listaron los **circuitos con mayor probabilidad de lluvia promedio**.
- Se identificaron las **escuderías con más campeonatos acumulados** entre sus pilotos.
- Se calculó un indicador nuevo de **eficiencia (puntos por millón de dólares de presupuesto)**,
  cruzando esta consulta SQL con la tabla `resumen_escuderia` que dejamos en el notebook 03.